# Vizualizacija podatkov — A: prikaz (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 1, Vizualizacija podatkov: kako izbrati pravi graf?**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_vizualizacija`** (telekom).

### Ponovitev
Graf izberemo glede na **vprašanje** in **tip spremenljivk**:

| Vprašanje | Spremenljivke | Graf |
|---|---|---|
| Kako so vrednosti porazdeljene? | 1 numerična | histogram + KDE, škatla z brki |
| Kako pogoste so kategorije? | 1 kategorična | stolpični diagram |
| Ali sta povezani? | 2 numerični | razsevni diagram |
| Kako se spreminja v času? | čas + numerična | črtni graf |
| Ali se skupine razlikujejo? | kategorična + numerična | škatle / violine po skupinah |
| Kaj je povezano s čim? | več numeričnih | korelacijska matrika; barva in podgrafi |

Grafe rišemo s knjižnico **plotnine** — "slovnica grafov", enako kot ggplot2 v R.
Graf sestavimo iz **plasti**, ki jih seštejemo s `+`:
- `ggplot(tabela, aes(x=..., y=...))` — podatki in kateri stolpec gre na katero os (ali v barvo: `fill=`, `color=`),
- `geom_...()` — oblika: `geom_histogram`, `geom_boxplot`, `geom_col`, `geom_point`, `geom_line` …,
- `scale_...`, `coord_...`, `facet_...`, `labs(...)`, `theme(...)` — merila osi, podgrafi, naslovi, videz.

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu. Koda je skrita — odprete jo s klikom na celico.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

### Knjižnice in videz grafov
`from plotnine import *` uvozi vse sestavne dele grafov. Spodaj nastavimo še enoten videz vseh grafov.
Ista koda je v `utils/orodja.py` kot funkcija `nastavi_slog()` — v naslednjih zvezkih jo samo pokličemo.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from plotnine import *
from plotnine.exceptions import PlotnineWarning

# Barve, ki jih ločijo tudi barvno slepi
MODRA, ORANZNA, ZELENA, SIVA = "#2a78d6", "#eb6834", "#1baf7a", "#9a9890"

theme_set(theme_minimal(base_size=11)                         # bela podlaga, svetla mreža
          + theme(figure_size=(8, 4.5),                       # velikost grafa (v palcih)
                  plot_title=element_text(weight="bold", size=13),
                  panel_grid_minor=element_blank()))
warnings.filterwarnings("ignore", category=PlotnineWarning)    # skrijemo opozorila "Removed N rows ..."
warnings.filterwarnings("ignore", message="divide by zero")     # log(0) na logaritemski osi

### Tabela naročil
Tabele trgovine (naročila, postavke, izdelki, plačila, ocene, stranke) združimo v eno tabelo:
**ena vrstica = eno naročilo**. Uporabimo `merge` (kot JOIN v SQL) in `groupby` (seštejemo postavke naročila).
Koda je skrita, ker ni tema tega bloka — odprete jo s klikom na celico.
Ista koda je v `utils/orodja.py` kot funkcija `pripravi_narocila(DATA)`.

In [ ]:
#@title Priprava tabele naročil (kliknite za prikaz kode)
d = DATA / "trgovina"
narocila = pd.read_csv(d / "olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
postavke = pd.read_csv(d / "olist_order_items_dataset.csv")
izdelki = pd.read_csv(d / "olist_products_dataset.csv")
prevod = pd.read_csv(d / "product_category_name_translation.csv")
placila = pd.read_csv(d / "olist_order_payments_dataset.csv")
ocene = pd.read_csv(d / "olist_order_reviews_dataset.csv")
stranke = pd.read_csv(d / "olist_customers_dataset.csv")

# 1. postavke + izdelki (kategorija v angleščini, teža)
izdelki = izdelki.merge(prevod, on="product_category_name", how="left")
postavke = postavke.merge(izdelki[["product_id", "product_category_name_english", "product_weight_g"]],
                          on="product_id", how="left")
# 2. postavke seštejemo po naročilu
po_narocilu = postavke.groupby("order_id").agg(
    vrednost=("price", "sum"), postnina=("freight_value", "sum"), st_postavk=("order_item_id", "count"),
    teza_g=("product_weight_g", "sum"), kategorija=("product_category_name_english", "first")).reset_index()
# 3. glavno plačilo naročila (največji znesek) in ocena kupca
glavno_placilo = (placila.sort_values("payment_value", ascending=False).drop_duplicates("order_id")
                  .rename(columns={"payment_type": "nacin_placila", "payment_installments": "st_obrokov"})
                  [["order_id", "nacin_placila", "st_obrokov"]])
ocena = (ocene.drop_duplicates("order_id")[["order_id", "review_score"]]
         .rename(columns={"review_score": "ocena"}))
# 4. vse skupaj
narocila = (narocila
            .merge(stranke[["customer_id", "customer_state"]], on="customer_id", how="left")
            .merge(po_narocilu, on="order_id", how="left")
            .merge(glavno_placilo, on="order_id", how="left")
            .merge(ocena, on="order_id", how="left"))
narocila["dni_dostave"] = (narocila["order_delivered_customer_date"]
                           - narocila["order_purchase_timestamp"]).dt.days
narocila["zamuda_dni"] = (narocila["order_delivered_customer_date"]
                          - narocila["order_estimated_delivery_date"]).dt.days
narocila["ocena"] = narocila["ocena"].astype("Int64")
narocila = narocila.rename(columns={"order_status": "status", "order_purchase_timestamp": "datum_nakupa",
                                    "customer_state": "zvezna_drzava"})
narocila = narocila[["order_id", "datum_nakupa", "status", "zvezna_drzava", "kategorija", "st_postavk",
                     "vrednost", "postnina", "teza_g", "nacin_placila", "st_obrokov",
                     "dni_dostave", "zamuda_dni", "ocena"]]
print("Tabela naročil:", narocila.shape)

In [ ]:
narocila.head()

---
# Del A — Prikaz: spletna trgovina

## A1. Ena numerična spremenljivka: histogram + KDE
Kje je glavnina podatkov? Je porazdelitev simetrična ali ima dolg rep? Ima en vrh ali več?

`after_stat("density")` nariše višino stolpcev kot gostoto (namesto števila), da se ujema s krivuljo KDE.

In [ ]:
(ggplot(narocila, aes(x="vrednost"))
 + geom_histogram(aes(y=after_stat("density")), bins=60, fill=MODRA)
 + geom_density()
 + labs(title="Vrednost naročila", x="vrednost (BRL)", y="gostota"))

Dolg desni rep: večina naročil je majhnih, nekaj je zelo velikih — na linearni osi ne vidimo ničesar.
**Logaritemska os** rep "stisne". Dodamo samo eno plast, `scale_x_log10()`.
Oznake na osi: `1e1` = 10, `1e2` = 100, `1e3` = 1000 (vsaka oznaka je 10-krat večja).

In [ ]:
(ggplot(narocila, aes(x="vrednost"))
 + geom_histogram(aes(y=after_stat("density")), bins=60, fill=MODRA)
 + geom_density()
 + scale_x_log10()
 + labs(title="Vrednost naročila — logaritemska os", x="vrednost (BRL)", y="gostota"))

## A2. Ena numerična spremenljivka: škatla z brki (boxplot)
Škatla = srednjih 50 % podatkov (od Q1 do Q3), črta v škatli = **mediana**.
Brki segajo do 1,5 × IQR, točke zunaj brkov so **kandidati za osamelce** (popoldne!).

V plotnine je škatla navpična; `coord_flip()` jo obrne vodoravno.

In [ ]:
(ggplot(narocila, aes(x="''", y="dni_dostave"))       # x="''": ena sama škatla, brez skupin
 + geom_boxplot(width=0.4, outlier_alpha=0.3)
 + coord_flip()
 + labs(title="Koliko dni traja dostava?", x="", y="dni dostave")
 + theme(figure_size=(8, 3)))

In [ ]:
narocila["dni_dostave"].describe()

Histogram in škatlo pogosto pogledamo skupaj. Operator `|` postavi dva grafa drug ob drugega.
To zapakiramo v **funkcijo** — uporabili jo bomo v vaji in v popoldanskih zvezkih (kopija je v `utils/orodja.py`):

In [ ]:
def histogram_in_skatla(df, stolpec, log=False):
    """Histogram s KDE (levo) in škatla z brki (desno) za en numerični stolpec."""
    podatki = df[df[stolpec].notna()]
    if log and (podatki[stolpec] <= 0).any():  # na logaritemski osi ni prostora za 0 in negativne vrednosti
        print(f"Na logaritemski osi ne prikažemo {(podatki[stolpec] <= 0).sum()} vrstic z vrednostjo <= 0.")
        podatki = podatki[podatki[stolpec] > 0]
    histogram = (ggplot(podatki, aes(x=stolpec))
                 + geom_histogram(aes(y=after_stat("density")), bins=50, fill=MODRA)
                 + geom_density()
                 + labs(y="gostota"))
    skatla = (ggplot(podatki, aes(x="''", y=stolpec))
              + geom_boxplot(width=0.4, outlier_alpha=0.3)
              + coord_flip()
              + labs(x=""))
    if log:
        histogram = histogram + scale_x_log10()
        skatla = skatla + scale_y_log10()
    return (histogram | skatla) + theme(figure_size=(11, 4))

histogram_in_skatla(narocila, "vrednost", log=True)

## A3. Ena kategorična spremenljivka: stolpični diagram
`value_counts()` prešteje kategorije, `geom_col` nariše stolpce, `scale_x_discrete(limits=...)` jih uredi po velikosti.
Manjkajoče vrednosti (`NaN`) `value_counts()` sicer izpusti — zato jih najprej poimenujemo `"manjka"`.

In [ ]:
stevila = (narocila["nacin_placila"]
           .fillna("manjka")              # manjkajoče pokažemo kot svojo kategorijo
           .value_counts()                # koliko naročil v vsaki kategoriji
           .reset_index())                # tabela s stolpcema: nacin_placila, count
print(stevila)

(ggplot(stevila, aes(x="nacin_placila", y="count"))
 + geom_col(fill=MODRA)
 + scale_x_discrete(limits=stevila["nacin_placila"][::-1].tolist())   # največja kategorija zgoraj
 + coord_flip()
 + labs(title="Način plačila", x="", y="Število naročil"))

Eno naročilo nima zapisa o plačilu — to je **manjkajoč podatek, ne plačilo 0**.

Zakaj ne tortni diagram? Oko slabo primerja kote in ploščine. Pri več kot ~3 kategorijah majhnih rezin ne ločimo.
plotnine tortnih diagramov sploh ne riše — za primerjavo jo narišemo z Matplotlibom:

In [ ]:
import matplotlib.pyplot as plt

plt.pie(stevila["count"], labels=stevila["nacin_placila"])
plt.title("Torta: koliko je 'debit_card'?")
plt.show()

Pri veliko kategorijah pokažemo samo najpogostejše. Kodo od zgoraj zapakiramo v **funkcijo** `graf_pogostosti`
(kopija je v `utils/orodja.py`). Kategorijo `manjka` pokaže vedno, tudi če ni med najpogostejšimi.

In [ ]:
def graf_pogostosti(df, stolpec, top=None):
    """Vodoravni stolpci: kolikokrat se pojavi vsaka vrednost. Manjkajoče pokaže kot 'manjka'."""
    stevila = (df[stolpec].astype("string")       # kot besedilo: deluje tudi za številske kode
               .fillna("manjka").value_counts().reset_index())
    if top:                                   # samo najpogostejše, 'manjka' pa vedno pokažemo
        stevila = stevila[(stevila.index < top) | (stevila[stolpec] == "manjka")]
    vrstni_red = stevila[stolpec][::-1].tolist()          # največja kategorija zgoraj
    return (ggplot(stevila, aes(x=stolpec, y="count"))
            + geom_col(fill=MODRA)
            + scale_x_discrete(limits=vrstni_red)
            + coord_flip()
            + labs(title=f"Pogostost: {stolpec}", x="", y="Število vrstic"))

graf_pogostosti(narocila, "kategorija", top=10)

## A4. Dve numerični spremenljivki: razsevni diagram
Ali so težja naročila dražja za pošiljanje? Pri 100 000 točkah se točke prekrivajo — zato vzamemo **vzorec**
in **prosojnost** (`alpha`).

In [ ]:
print(f"Korelacija teža–poštnina (vsa naročila): r = {narocila['teza_g'].corr(narocila['postnina']):.2f}")
vzorec = narocila.dropna(subset=["teza_g", "postnina"]).sample(3000, random_state=42)

(ggplot(vzorec, aes(x="teza_g", y="postnina"))
 + geom_point(alpha=0.4, size=1.5, color=MODRA)
 + labs(title="Poštnina glede na težo naročila", x="teža (g)", y="poštnina (BRL)"))

Težja naročila imajo praviloma dražjo poštnino (r ≈ 0,6), a raztros je velik: poštnina je odvisna tudi od razdalje.

## A5. Čas: črtni graf
Koliko naročil na teden? Najprej preverimo, koliko naročil je na **začetku in koncu** podatkov:

In [ ]:
na_mesec = narocila["datum_nakupa"].dt.to_period("M").value_counts().sort_index()
print(na_mesec.head(5).to_string())
print("...")
print(na_mesec.tail(5).to_string())

Na robovih je zelo malo naročil: zajem podatkov se je šele začel oziroma končal. Če robove pustimo, graf pokaže
lažen padec. Zato jih **odrežemo**: `loc` izbere obdobje, `resample("W")` prešteje naročila po tednih.

In [ ]:
tedensko = (narocila.set_index("datum_nakupa").sort_index()
            .loc["2017-01-01":"2018-08-19"]
            .resample("W").size()
            .reset_index(name="narocil"))            # tabela s stolpcema: datum_nakupa, narocil
vrh = tedensko.loc[tedensko["narocil"].idxmax()]
print(f"Največ naročil ({vrh['narocil']}) v tednu, ki se konča {vrh['datum_nakupa'].date()}. "
      "Black Friday 2017 je bil 24. 11. 2017.")

(ggplot(tedensko, aes(x="datum_nakupa", y="narocil"))
 + geom_line(color=MODRA, size=1)
 + annotate("text", x=vrh["datum_nakupa"], y=vrh["narocil"], label="  Black Friday 2017", ha="left")
 + labs(title="Število naročil na teden", x="", y="naročil")
 + theme(figure_size=(11, 4)))

## A6. Kategorična + numerična: škatle po skupinah
Ali kupci, ki so dlje čakali na dostavo, dajejo slabše ocene?
`factor(ocena)` pove, da je ocena **kategorija** (5 škatel), ne število.
`outlier_shape=""` skrije točke osamelcev, `coord_cartesian(ylim=...)` poveča območje do 60 dni, ne da bi podatke brisal.

In [ ]:
ocenjena = narocila.dropna(subset=["ocena", "dni_dostave"])

(ggplot(ocenjena, aes(x="factor(ocena)", y="dni_dostave"))
 + geom_boxplot(outlier_shape="")
 + coord_cartesian(ylim=(0, 60))
 + labs(title="Dni dostave glede na oceno kupca", x="ocena", y="dni dostave"))

In [ ]:
ocenjena.groupby("ocena")["dni_dostave"].median()

Violina pokaže še obliko porazdelitve znotraj vsake skupine (črte: Q1, mediana, Q3):

In [ ]:
(ggplot(ocenjena[ocenjena["dni_dostave"] < 60], aes(x="factor(ocena)", y="dni_dostave"))
 + geom_violin(fill=MODRA, alpha=0.5, draw_quantiles=[0.25, 0.5, 0.75])
 + labs(title="Dni dostave glede na oceno kupca (violina)", x="ocena", y="dni dostave"))

## A7. Več spremenljivk: korelacijska matrika
Korelacija meri **linearno** povezanost med −1 in +1. Korelacija **ni vzročnost**.

Za plotnine matriko najprej pretvorimo v **dolgo obliko** (`melt`): ena vrstica = en par stolpcev.
To je tipičen korak pri plotnine: en stolpec za os x, en za os y, en za barvo.

In [ ]:
stolpci = ["vrednost", "postnina", "teza_g", "st_postavk", "st_obrokov", "dni_dostave", "zamuda_dni", "ocena"]
kor = narocila[stolpci].corr()

dolga = (kor.round(2).rename_axis("stolpec_1").reset_index()
         .melt(id_vars="stolpec_1", var_name="stolpec_2", value_name="r"))
print(dolga.head(3))

(ggplot(dolga, aes(x="stolpec_1", y="stolpec_2", fill="r"))
 + geom_tile(color="white")
 + geom_text(aes(label="r"), size=8)
 + scale_fill_gradient2(low=MODRA, mid="#f0efec", high=ORANZNA, limits=(-1, 1))   # modra (−1), siva (0), oranžna (+1)
 + scale_x_discrete(limits=stolpci)
 + scale_y_discrete(limits=stolpci[::-1])
 + coord_equal()
 + theme(figure_size=(6.5, 5.5), axis_text_x=element_text(rotation=45, ha="right"))
 + labs(title="Korelacijska matrika", x="", y=""))

Za vajo to kodo zapakiramo v funkcijo `graf_korelacij(kor)` (kopija je v `utils/orodja.py`):

In [ ]:
def graf_korelacij(kor):
    """Korelacijska matrika: modra = negativna, oranžna = pozitivna korelacija."""
    dolga = (kor.round(2).rename_axis("stolpec_1").reset_index()
             .melt(id_vars="stolpec_1", var_name="stolpec_2", value_name="r"))
    imena = kor.columns.tolist()
    n = len(imena)
    return (ggplot(dolga, aes(x="stolpec_1", y="stolpec_2", fill="r"))
            + geom_tile(color="white")
            + geom_text(aes(label="r"), size=8 if n <= 10 else 6)
            + scale_fill_gradient2(low=MODRA, mid="#f0efec", high=ORANZNA, limits=(-1, 1))
            + scale_x_discrete(limits=imena)
            + scale_y_discrete(limits=imena[::-1])
            + coord_equal()
            + theme(figure_size=(max(6, 0.6 * n), max(5, 0.55 * n)),
                    axis_text_x=element_text(rotation=45, ha="right"))
            + labs(title="Korelacijska matrika", x="", y=""))

## A8. Tretja spremenljivka: barva in podgrafi
Z `fill=` ali `color=` v `aes` dodamo kategorično spremenljivko kot **barvo**.
Ali nezadovoljni kupci (ocena 1–2) dlje čakajo na dostavo?

In [ ]:
ocenjena = ocenjena.assign(kupec=np.where(ocenjena["ocena"] <= 2, "nezadovoljen (1–2)", "zadovoljen (3–5)"))
print(ocenjena.groupby("kupec")["dni_dostave"].median())

(ggplot(ocenjena[ocenjena["dni_dostave"] < 60], aes(x="dni_dostave", fill="kupec"))
 + geom_density(alpha=0.5)
 + scale_fill_manual(values=[ORANZNA, MODRA])
 + labs(title="Dni dostave: nezadovoljni in zadovoljni kupci", x="dni dostave", y="gostota", fill=""))

**Podgrafi** (`facet_wrap`): isti graf narišemo za vsako skupino posebej — tu histogram za vsako oceno.
`scales="free_y"`: vsak podgraf ima svojo os y, ker so skupine različno velike.

In [ ]:
(ggplot(ocenjena[ocenjena["dni_dostave"] < 60], aes(x="dni_dostave"))
 + geom_histogram(binwidth=2, fill=MODRA)
 + facet_wrap("~ocena", ncol=5, scales="free_y")
 + labs(title="Dni dostave po ocenah kupca (1–5)", x="dni dostave", y="število naročil")
 + theme(figure_size=(11, 3.5)))

Pri ocenah 1 in 2 je desni rep (dolge dostave) debelejši. To bo kasneje model "videl" sam — iz vseh stolpcev hkrati.

---
Naprej: zvezek **`B_vizualizacija`** — iste grafe narišete na podatkih telekoma.